# Feature engineering research
All indicators below come from `feature_engineering.indicators`; this notebook only inspects them. Calculations are causal and isolated by symbol.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

In [1]:
import altair as alt
import pandas as pd

from data_pipeline.src.config import MASTER_CSV_PATH
from feature_engineering.indicators import calculate_features
from feature_engineering.schemas import FEATURE_COLUMNS, FEATURE_VERSION

market = pd.read_csv(MASTER_CSV_PATH, dtype={"symbol": "string"})
selected_symbol = "MCB" if "MCB" in set(market["symbol"]) else str(market["symbol"].iloc[0])
symbol_history = market.loc[market["symbol"] == selected_symbol]
featured = calculate_features(symbol_history)
print("Feature version:", FEATURE_VERSION)

ModuleNotFoundError: No module named 'data_pipeline'

In [ ]:
featured[["symbol", "date", *FEATURE_COLUMNS, "is_warmup"]].tail()

In [ ]:
moving_averages = featured[["date", "close", "sma_20", "sma_50"]].melt(
    id_vars="date", var_name="series", value_name="price"
)
alt.Chart(moving_averages).mark_line().encode(
    x=alt.X("date:T", title="Trading date"),
    y=alt.Y("price:Q", title="Price (PKR)"),
    color=alt.Color("series:N", title="Series"),
).properties(title=f"{selected_symbol}: close and moving averages", width=800, height=350)

In [ ]:
featured[["date", "rsi_14", "macd", "macd_signal", "macd_histogram", "rolling_volatility_20"]].tail(20)

## Warm-up interpretation
The first 49 observations per symbol are marked `is_warmup=True` because SMA/EMA 50 is unavailable. Values remain missing; they are never backfilled from future observations or replaced with zero. With the current short local history, there may be no fully warmed-up rows.

In [ ]:
featured.groupby("is_warmup", dropna=False).size().rename("rows")